In [1]:
import torch
import torch.nn as nn
import numpy as np

In [2]:
import seaborn as sns
iris = sns.load_dataset("iris")

data = torch.tensor( iris[iris.columns[0:4]].values ).float()

labels = torch.zeros( len(data), dtype=torch.long )
labels[iris.species=="versicolor"] = 1
labels[iris.species=="virginica"] = 2

In [17]:
propTraining = .8
nTraining = int(len(labels)*propTraining)

traintestBool = np.zeros(len(labels), dtype=bool)

# traintestBool[range(nTraining)] = True
items2use4train = np.random.choice(range(len(labels)), nTraining, replace=False)
traintestBool[items2use4train] = True

traintestBool

array([ True, False,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True, False, False,
       False, False,  True,  True,  True,  True,  True,  True,  True,
        True, False,  True,  True,  True,  True, False, False,  True,
        True,  True,  True,  True,  True, False,  True,  True,  True,
        True, False,  True,  True, False,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True, False,  True,
        True,  True, False,  True,  True,  True, False, False, False,
        True, False,  True, False,  True,  True,  True,  True, False,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
       False, False,  True,  True,  True,  True,  True, False, False,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True, False, False, False,  True,  True,  True,
        True,  True,

In [18]:
print("Average of full data: ")
print( torch.mean(labels.float()), end="\n\n")

print("Average of training data: ")
print( torch.mean(labels[traintestBool].float()), end="\n\n")

print("Average of test data: ")
print( torch.mean(labels[~traintestBool].float()) )

Average of full data: 
tensor(1.)

Average of training data: 
tensor(0.9833)

Average of test data: 
tensor(1.0667)


In [19]:
ANNiris = nn.Sequential(
    nn.Linear(4, 64),
    nn.ReLU(),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 3),
)

lossfun = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(ANNiris.parameters(), lr=0.01)

In [20]:
data.shape

torch.Size([150, 4])

In [21]:
data[traintestBool, :].shape

torch.Size([120, 4])

In [22]:
data[~traintestBool, :].shape

torch.Size([30, 4])

In [26]:
numepochs = 10000

losses = torch.zeros(numepochs)
ongoingAcc = []

for epochi in range(numepochs):

    yHat = ANNiris( data[traintestBool, :] )

    ongoingAcc.append( 100*torch.mean(
    (torch.argmax(yHat, axis=1) == labels[traintestBool]).float())
                     )
    loss = lossfun(yHat, labels[traintestBool])
    losses[epochi] = loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [27]:
predictions = ANNiris( data[traintestBool, :] )
trainacc = 100*torch.mean( (torch.argmax(predictions, axis=1) == labels[traintestBool]).float() )

predictions = ANNiris( data[~traintestBool, :] )
testacc = 100*torch.mean((torch.argmax(predictions,axis=1) == labels[~traintestBool]).float())

In [28]:
print("Final TRAIN accuracy: ", trainacc)
print("Final TEST accuracu: ", testacc)

Final TRAIN accuracy:  tensor(99.1667)
Final TEST accuracu:  tensor(90.)
